# Running Evaluation Benchmarks

### We will be running an evaluation benchmark provided with CHAMMI-75 dataset as an example. The benchmark we will be running is the RBC-MC benchmark, which evaluates the performance of a multi-class classification model on red blood cells morphology.

Library Imports

In [14]:
import pandas as pandas
import boto3
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from botocore import UNSIGNED
from botocore.config import Config
import os
import sys
sys.path.append("../benchmarks/rbc-mc")
sys.path.append("../benchmarks/")

Downloading the RBC-MC Benchmark from AWS S3 Bucket

In [7]:
!mkdir -p rbc-mc

In [9]:
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

def download_file(args):
    bucket, key, local_path = args
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    s3.download_file(bucket, key, local_path)

# List all objects first
paginator = s3.get_paginator('list_objects_v2')
files = []
for page in paginator.paginate(Bucket='chammi-data', Prefix='CHAMMI-75/CHAMMI-75_test/rbc-mc/'):
    for obj in page.get('Contents', []):
        key = obj['Key']
        local_path = key.replace('CHAMMI-75/CHAMMI-75_test/rbc-mc/', 'rbc-mc/')
        files.append(('chammi-data', key, local_path))

# Download with progress bar
with ThreadPoolExecutor(max_workers=64) as executor:
    list(tqdm(executor.map(download_file, files), total=len(files)))

100%|██████████| 130008/130008 [09:23<00:00, 230.75it/s]


After downloading the dataset, we will extract features and then run the benchmark evaluation script.

In [20]:
!accelerate launch --num_processes=1 ../benchmarks/rbc-mc/extraction.py --model huggingface_chammi75 --image_folder rbc-mc --output_folder rbc-mc/

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
/scr/vidit/CHAMMI-75/benchmarks/feat_models/multi_channel_vit.py:151: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
Total samples in dataset: 130008
Some weights of the model checkpoint at CaicedoLab/CHAMMI-75 were not used when initializing VisionTransformer: ['head.last_layer.parametrizations.weight.original0', 'head.last_layer.parametrizations.weight.original1', 'head.mlp.0.bias', 'head.mlp.0.weight', '

In [21]:
!python ../benchmarks/rbc-mc/regression.py --output_folder rbc-mc/ --pkl_path rbc-mc/embeddings.pkl

CROSS-DATASET LOGISTIC REGRESSION CLASSIFICATION
Vision Transformer Embeddings (384-dim)

⚠️  Excluding classes: Undecidable

[Step 1/6] Extracting data from both datasets...
  ✓ Swiss dataset: 76206 samples × 384 features
  ✓ Canadian dataset: 46542 samples × 384 features

[Step 2/6] Preparing label encoder...
  ✓ Number of classes: 7
  ✓ Classes: CrenatedDisc, CrenatedDiscoid, CrenatedSphere, CrenatedSpheroid, Side, SmoothDisc, SmoothSphere

[Step 3/6] Experiment 1: Train on Canadian → Validate on Swiss
  Training model...
  Making predictions...
  ✓ Accuracy: 0.6831 (68.31%)

[Step 4/6] Experiment 2: Train on Swiss → Validate on Canadian
  Training model...
  Making predictions...
  ✓ Accuracy: 0.6505 (65.05%)

[Step 5/6] Creating confusion matrix visualization...
Figure(1600x600)

SUMMARY

Experiment 1 (Train Canadian → Test Swiss):
  Accuracy: 68.3%

Experiment 2 (Train Swiss → Test Canadian):
  Accuracy: 65.1%

Average Cross-Dataset Accuracy: 66.7%

